In [ ]:
from PIL import Image
import os

def resize_images(input_folder, output_folder, new_size=(460, 460)):
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
    for filename in os.listdir(input_folder):
        if filename.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.gif')):
            img_path = os.path.join(input_folder, filename)
            img = Image.open(img_path)
            img_resized = img.resize(new_size, Image.LANCZOS)
            img_resized.save(os.path.join(output_folder, filename))

# Ejemplo de uso:
# resize_images('carpeta_original', 'carpeta_reducida', (460, 460))

In [3]:
import os
import numpy as np
from PIL import Image

d_original = '/home/smoke/bandas/banda13/resize'
d_reducida = '/home/smoke/bandas/banda13/reducida'

def resize_npy_images(d_original, d_reducida, new_size=(460, 460)):
    if not os.path.exists(d_reducida):
        os.makedirs(d_reducida)
    for filename in os.listdir(d_original):
        if filename.lower().endswith('.npy'):
            img_path = os.path.join(d_original, filename)
            img_array = np.load(img_path)
            img_resized = np.array(Image.fromarray(img_array).resize(new_size, Image.LANCZOS))
            np.save(os.path.join(d_reducida, filename), img_resized)

resize_npy_images(d_original, d_reducida, (460, 460))

In [6]:
%pip install tensorflow

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, Flatten, Dense
import numpy as np
import os

# Directorio de imágenes reducidas
d_reducida = '/home/smoke/bandas/banda13/reducida'

# Cargar imágenes .npy y normalizar según el rango real de radiancia
image_list = []
for filename in os.listdir(d_reducida):
    if filename.lower().endswith('.npy'):
        img = np.load(os.path.join(d_reducida, filename))
        # Normalización robusta: escala entre 0 y 1 usando el rango de radiancia real
        img = (img - np.min(img)) / (np.max(img) - np.min(img) + 1e-8)
        image_list.append(img)

X = np.stack(image_list)
# Añadir canal si es necesario (por ejemplo, imágenes en escala de grises)
if X.ndim == 3:
    X = X[..., np.newaxis]

# Modelo simple con una capa Conv2D
model = Sequential([
    Conv2D(16, (3, 3), activation='relu', input_shape=X.shape[1:]),
    Flatten(),
    Dense(1, activation='sigmoid')  # Cambia según tu tarea
])

model.compile(optimizer='adam', loss='binary_crossentropy')

# Para entrenar necesitas etiquetas (y), aquí solo se muestra cómo preparar X
# model.fit(X, y, epochs=10)

  Using cached setuptools-80.9.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached rich-14.1.0-py3-none-any.whl.metadata (18 kB)
  Using cached idna-3.10-py3-none-any.whl.metadata (10 kB)
  Using cached urllib3-2.5.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached MarkupSafe-3.0.2-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (4.0 kB)
  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 620.7/620.7 MB 12.9 MB/s eta 0:00:00m eta 0:00:010:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 46.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.4/6.4 MB 106.0 MB/s eta 0:00:00 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 94.5 MB/s eta 0:00:006 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 96.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.5/

2025-09-16 18:57:44.347524: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2025-09-16 18:57:44.347779: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-09-16 18:57:44.380868: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-09-16 18:57:45.182720: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off,